# Light or Letdown? — Social Media Analysis of the **Ferrari Luce**
### Web and Social Media Search and Analysis — BSc AI, UniMiB

Self-contained Colab notebook for the Track-1 project (Network + Content + Visualization).
It writes the tested project code (`src/`) into Colab and runs the full pipeline end-to-end,
**including the Reddit scraping** (LAB 1–2).

**Runs with no credentials.** Reddit is scraped live via **Arctic-Shift → PullPush**
(no account, no key); Bluesky is included only if you add an App Password in §2. The full
Reddit pull takes ~10–15 min.

**How to use:** *Runtime → Change runtime type → GPU* (recommended — the transformer models
auto-detect and use it) → **Runtime → Run all**. Pipeline: collect → graph (L3) → communities
(L4) → sentiment/ABSA (L5) → sarcasm/stance/topics/language → NER (L6) → visualization →
download everything. See `MEGAPLAN.md`.

In [1]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.cuda.empty_cache()
else:
    device = torch.device('cpu')
    print('No GPU found — running on CPU. Go to Runtime → Change runtime type → GPU.')

print(f'Using device: {device}')

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB
Using device: cuda


## 1. Install dependencies
Colab bundles torch/transformers/pandas/sklearn/networkx/nltk/matplotlib/spaCy. Install the rest.

In [2]:
# Core pipeline deps. Colab already ships torch / transformers / pandas /
# scikit-learn / networkx / nltk / spaCy / matplotlib / seaborn / wordcloud / requests.
# NRCLex needs textblob for its load_raw_text() tokeniser (NRC emotions, LAB 5).
#!pip install -q atproto praw vaderSentiment afinn NRCLex textblob langdetect python-louvain python-dotenv

# BERTopic is an optional extension; if its install fails the pipeline falls
# back automatically to a scikit-learn LDA topic model (no action needed).
#pip install -q bertopic || echo "bertopic not installed -> sklearn LDA fallback will be used"

# spaCy English model for NER (LAB 6)
#!python -m spacy download en_core_web_sm -q
#print("dependencies installed.")

In [3]:
import nltk
for pkg in ("punkt","punkt_tab","stopwords","wordnet","omw-1.4"):
    nltk.download(pkg, quiet=True)
print("NLTK data ready.")

NLTK data ready.


## 2. Credentials — all OPTIONAL (the notebook runs with none)
- **Reddit:** leave blank → collected via **PullPush.io** (no account, no key, no OAuth).
- **Bluesky:** leave blank → Bluesky is **skipped**; to include it, add an *App Password* (bsky.app → Settings → App Passwords).

Nothing is stored. Best practice: use Colab **Secrets** (🔑 in the left sidebar) named `BLUESKY_HANDLE`, `BLUESKY_APP_PASSWORD`, `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET` — the next cell reads them automatically. Otherwise paste values into the cell. Either way **Runtime → Run all** completes unattended.

In [4]:
import os

# ── Optional credentials ── leave blank to run with NO account ──────────────
# Reddit blank -> PullPush.io (no account).  Bluesky blank -> Bluesky skipped.
# Paste here (kept in memory only) OR, preferably, set them as Colab Secrets.
BLUESKY_HANDLE       = "steffenrizzuto.bsky.social"   # e.g. "you.bsky.social"
BLUESKY_APP_PASSWORD = "rizzuto6"   # App Password, NOT your main password
REDDIT_CLIENT_ID     = ""   # leave blank -> PullPush.io
REDDIT_CLIENT_SECRET = ""   # leave blank -> PullPush.io
REDDIT_USERNAME      = "student"   # only used to build a polite user-agent string

def _resolve(name, pasted):
    """Pasted value > Colab Secret > existing environment variable > ''."""
    if pasted:
        return pasted
    try:
        from google.colab import userdata          # only exists on Colab
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

os.environ["BLUESKY_HANDLE"]       = _resolve("BLUESKY_HANDLE", BLUESKY_HANDLE)
os.environ["BLUESKY_APP_PASSWORD"] = _resolve("BLUESKY_APP_PASSWORD", BLUESKY_APP_PASSWORD)
os.environ["REDDIT_CLIENT_ID"]     = _resolve("REDDIT_CLIENT_ID", REDDIT_CLIENT_ID)
os.environ["REDDIT_CLIENT_SECRET"] = _resolve("REDDIT_CLIENT_SECRET", REDDIT_CLIENT_SECRET)
os.environ["REDDIT_USER_AGENT"]    = f"wsa-ferrari-luce/0.1 by u/{REDDIT_USERNAME or 'student'}"

print("Bluesky:", "ON" if os.environ["BLUESKY_HANDLE"] else "skipped (no creds)")
print("Reddit :", "PRAW (API)" if os.environ["REDDIT_CLIENT_ID"] else "PullPush.io (no account)")

Bluesky: ON
Reddit : PullPush.io (no account)


## 3. Project code
This notebook **imports the tested package in `src/`** directly. If you opened the project
folder (uploaded to Colab or mounted from Drive) `src/` is already here; on a bare Colab the
cell below clones the repo to fetch it.

In [5]:
import os, subprocess
# Make the src/ package available. If you're running inside the project folder,
# src/ is already here; otherwise clone the repo to fetch it.
if not os.path.isdir("src"):
    subprocess.run(
        "git clone --depth 1 https://github.com/akirykouski/wsa-ferrari-luce.git _wsa "
        "&& cp -r _wsa/src ./src && cp -r _wsa/report ./report 2>/dev/null",
        shell=True,
    )
if not os.path.isdir("src"):
    raise RuntimeError(
        "src/ not found. The repo is private, so either run this notebook from the "
        "project folder (upload it to Colab or mount Google Drive), or clone a repo "
        "you can access."
    )
for d in ("data/raw", "data/processed", "figures", "report"):
    os.makedirs(d, exist_ok=True)
print("project code ready:", os.path.isdir("src"))

project code ready: True


## 4. Run the pipeline

In [6]:
import pandas as pd
from IPython.display import Image, display
import glob

### 4.1 Collect the data (LAB 1–2)
Scrapes the real data **live**: the official Reddit API if you entered creds in §2,
otherwise **Arctic-Shift → PullPush** (no account, no key). Bluesky is included only if you
supplied an App Password. The full Reddit pull (~370 submissions + their comment trees)
takes **~10–15 min**; re-running fetches fresh posts.

In [7]:
# ── Collect the data (LAB 1-2) ──────────────────────────────────────────
# Live scrape: Reddit API if creds were given in §2, else Arctic-Shift ->
# PullPush (no account). Bluesky only if an App Password was supplied.
from src.collect_bluesky import main as run_bluesky
from src.collect_reddit import main as run_reddit

try:
    run_bluesky()
except Exception as e:
    print("Bluesky collection skipped:", e)

run_reddit()

2026-06-05 15:34:07,900 INFO HTTP Request: POST https://bsky.social/xrpc/com.atproto.server.createSession "HTTP/1.1 200 OK"
2026-06-05 15:34:08,460 INFO HTTP Request: GET https://stropharia.us-west.host.bsky.network/xrpc/app.bsky.actor.getProfile?actor=steffenrizzuto.bsky.social "HTTP/1.1 200 OK"
2026-06-05 15:34:08,461 INFO Bluesky login OK as steffenrizzuto.bsky.social
2026-06-05 15:34:08,750 INFO HTTP Request: GET https://stropharia.us-west.host.bsky.network/xrpc/app.bsky.feed.searchPosts?q=%22Ferrari+Luce%22&limit=25&since=2025-10-01T00%3A00%3A00Z&sort=latest "HTTP/1.1 200 OK"
2026-06-05 15:34:09,400 INFO HTTP Request: GET https://stropharia.us-west.host.bsky.network/xrpc/app.bsky.feed.searchPosts?q=%22Ferrari+Luce%22&cursor=25&limit=25&since=2025-10-01T00%3A00%3A00Z&sort=latest "HTTP/1.1 200 OK"
2026-06-05 15:34:10,033 INFO HTTP Request: GET https://stropharia.us-west.host.bsky.network/xrpc/app.bsky.feed.searchPosts?q=%22Ferrari+Luce%22&cursor=50&limit=25&since=2025-10-01T00%3A00%

### 4.2 Network analysis — graph + centralities (LAB 3), communities (LAB 4)

In [8]:
from src.build_graph import main as run_graph
from src.communities import main as run_comm
run_graph(); run_comm()
print(open("data/processed/graph_summary.txt").read())
print(open("data/processed/communities_summary.txt").read())
display(pd.read_csv("data/processed/nodes_centrality.csv").head(15))

2026-06-05 15:49:16,882 INFO Interaction graph: 8223 nodes, 13792 edges
2026-06-05 15:49:29,631 INFO saved 8223 rows -> C:\Users\stefa\Downloads\wsa-ferrari-luce\data\processed\nodes_centrality.csv
2026-06-05 15:49:30,043 INFO Graph summary:
nodes: 8223
edges: 13792
density: 0.00020
transitivity (clustering): 0.0050
avg clustering: 0.0368
reciprocity: 0.2338
weakly connected components: 125
strongly connected components: 6149
degree assortativity: -0.1590
2026-06-05 15:49:30,614 INFO graph -> C:\Users\stefa\Downloads\wsa-ferrari-luce\data\processed\graph.graphml (open in Gephi)
2026-06-05 15:49:45,427 INFO saved 8223 rows -> C:\Users\stefa\Downloads\wsa-ferrari-luce\data\processed\nodes_communities.csv
2026-06-05 15:49:45,454 INFO Communities:
louvain_n_communities: 165
louvain_modularity: 0.759316090113547
greedy_n_communities: 162
greedy_modularity: 0.7565440468772843
degree_assortativity: -0.22109142643425117
min_community_size: 5
louvain_communities_ge_min: 53

Top members per Louv

nodes: 8223
edges: 13792
density: 0.00020
transitivity (clustering): 0.0050
avg clustering: 0.0368
reciprocity: 0.2338
weakly connected components: 125
strongly connected components: 6149
degree assortativity: -0.1590
louvain_n_communities: 165
louvain_modularity: 0.759316090113547
greedy_n_communities: 162
greedy_modularity: 0.7565440468772843
degree_assortativity: -0.22109142643425117
min_community_size: 5
louvain_communities_ge_min: 53

Top members per Louvain community (n>=5; name the camps from these):
  community 6 (n=28): ['thebishf1.bsky.social', 'thetastate.bsky.social', 'd2lo.bsky.social', 'msdianan.bsky.social', 'hrothger4hand.bsky.social', 'lestatdelc.bsky.social', 'arnoldziffel.bsky.social', 'cjwl.bsky.social']
  community 8 (n=9): ['niedermeyer.online', 'peark.es', 'thecrystaljules.bsky.social', 'horadam.bsky.social', 'munson.land', 'manitou202.bsky.social', 'nme365.bsky.social', 'dieterjosef.bsky.social']
  community 10 (n=8): ['dmeance.bsky.social', 'frandroid.com', 'nu

,node,platform,in_degree,out_degree,in_degree_centrality,out_degree_centrality,betweenness,closeness,pagerank,eigenvector
0,Public-Promotion-744,reddit,259,17,0.031501,0.002068,0.007764,0.084766,0.025062,NaN
1,k_entity,reddit,260,0,0.031622,0.000000,0.000000,0.154024,0.012820,NaN
2,NegotiationNew9264,reddit,305,6,0.037096,0.000730,0.011763,0.121039,0.011888,NaN
3,Mac-Tyson,reddit,252,12,0.030649,0.001459,0.017541,0.140776,0.011761,NaN
4,DeonEVE,reddit,363,0,0.044150,0.000000,0.000000,0.152122,0.010084,NaN
5,mightyopik,reddit,142,12,0.017271,0.001459,0.009722,0.125083,0.009846,NaN
6,Slice5755,reddit,38,7,0.004622,0.000851,0.004188,0.127022,0.008940,NaN
7,Fair_Title2995,reddit,173,81,0.021041,0.009852,0.059754,0.134763,0.007267,NaN
8,FoxHound6112,reddit,256,1,0.031136,0.000122,0.000012,0.132880,0.006279,NaN
9,Effective_Moose_4997,reddit,64,20,0.007784,0.002432,0.004221,0.077673,0.006209,NaN


### 4.3 Content — sentiment/emotion/ABSA (LAB 5) + enrichments (sarcasm RQ6, stance RQ7, topics, language RQ8)

In [9]:
from src.content_sentiment import main as run_sent
from src.content_enrich import main as run_enrich
run_sent(); run_enrich()
display(pd.read_csv("data/processed/aspect_sentiment.csv"))
display(pd.read_csv("data/processed/language_sentiment.csv"))

2026-06-05 15:49:45,904 INFO dropped 300 bot-authored documents
2026-06-05 15:50:35,282 INFO Corpus: 19680 documents ({'reddit': 16992, 'bluesky': 2688})
2026-06-05 15:50:55,397 INFO NRC emotions: 12134/19680 docs matched the lexicon
c:\Users\stefa\Downloads\wsa-ferrari-luce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-05 15:51:05,145 INFO HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-05 15:51:05,158 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-roberta-base-sentiment-latest/3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7/config.json "HTTP/1.1 200 OK"
2026-06-05 15:51:05,310 INFO HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-roberta-ba

,aspects,negative,neutral,positive,total,negative_ratio
0,electrification,3289,3054,1456,7799,0.422
1,exterior_look,2853,1945,1238,6036,0.473
2,design_house,1474,1568,689,3731,0.395
3,name,818,1887,568,3273,0.250
4,price,1338,1077,446,2861,0.468
5,performance,572,707,332,1611,0.355
6,interior,528,527,384,1439,0.367
7,brand_identity,644,561,187,1392,0.463
8,sound,174,162,82,418,0.416


,lang_group,negative,neutral,positive,total,negative_ratio
0,en,8031,6718,2216,16965,0.473
1,it,13,394,15,422,0.031
2,other,375,1730,188,2293,0.164


### 4.4 NER + entity-level sentiment (LAB 6)

In [10]:
from src.ner_entities import main as run_ner
run_ner()
display(pd.read_csv("data/processed/entity_frequency.csv").head(20))
display(pd.read_csv("data/processed/entity_sentiment.csv").head(20))

2026-06-05 16:13:50,463 INFO saved 6553 rows -> C:\Users\stefa\Downloads\wsa-ferrari-luce\data\processed\entity_frequency.csv
2026-06-05 16:13:50,519 INFO saved 2562 rows -> C:\Users\stefa\Downloads\wsa-ferrari-luce\data\processed\entity_cooccurrence.csv
2026-06-05 16:13:50,889 INFO saved 6553 rows -> C:\Users\stefa\Downloads\wsa-ferrari-luce\data\processed\entity_sentiment.csv
2026-06-05 16:13:50,896 INFO Top entities:
      entity   label  count
     ferrari  PERSON   5052
        luce PRODUCT    722
ferrari luce     ORG    507
     chinese    NORP    379
       apple     ORG    352
     porsche     ORG    312
        jony  PERSON    290
    ferraris  PERSON    271
       china     GPE    210
      taycan    NORP    208
       tesla    NORP    197
     italian    NORP    170
       honda     ORG    137
         bmw     ORG    119
         f80     ORG    113


,entity,label,count
0,ferrari,PERSON,5052
1,luce,PRODUCT,722
2,ferrari luce,ORG,507
3,chinese,NORP,379
4,apple,ORG,352
5,porsche,ORG,312
6,jony,PERSON,290
7,ferraris,PERSON,271
8,china,GPE,210
9,taycan,NORP,208


,entity,total,positive,neutral,negative,negative_ratio
0,ferrari,5829,935,2638,2256,0.387
1,luce,734,168,322,244,0.332
2,ferrari luce,515,66,381,68,0.132
3,chinese,379,38,135,206,0.544
4,apple,353,75,149,129,0.365
5,porsche,312,67,137,108,0.346
6,jony,290,77,125,88,0.303
7,ferraris,280,51,118,111,0.396
8,tesla,211,44,78,89,0.422
9,china,210,22,91,97,0.462


### 4.5 Visualizations
All charts are saved to `figures/` as PNGs, including the community-coloured interaction network.

In [1]:
from src.viz import main as run_viz
run_viz()
for f in sorted(glob.glob("figures/*.png")):
    print(f); display(Image(f))

2026-06-05 18:14:51,523 INFO figure -> C:\Users\stefa\Downloads\wsa-ferrari-luce\figures\sentiment_distribution.png
2026-06-05 18:14:51,747 INFO figure -> C:\Users\stefa\Downloads\wsa-ferrari-luce\figures\emotions.png
2026-06-05 18:14:51,928 INFO figure -> C:\Users\stefa\Downloads\wsa-ferrari-luce\figures\aspect_sentiment_stacked.png
2026-06-05 18:14:52,014 INFO figure -> C:\Users\stefa\Downloads\wsa-ferrari-luce\figures\aspect_share_pie.png
2026-06-05 18:14:52,331 INFO figure -> C:\Users\stefa\Downloads\wsa-ferrari-luce\figures\sentiment_timeline.png
c:\Users\stefa\Downloads\wsa-ferrari-luce\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['let'] not in stop_words.
  warnings.warn(
2026-06-05 18:14:53,978 INFO figure -> C:\Users\stefa\Downloads\wsa-ferrari-luce\figures\wordcloud_negative.png
2026-06-05 18:14:54,416 INFO figure -> C:\Users\stefa\Downloads

NameError: name 'glob' is not defined

### 4.6 Download everything
One Colab run produces the complete result set from the bundled data. This packs all
tables (`data/processed/`) and charts (`figures/`, including the interactive
`community_network.html`) into a single zip and downloads it — so you leave with everything.

In [ ]:
import os, zipfile
ZIP = "WSA_FerrariLuce_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root in ("data/processed", "figures"):
        for dp, _, files in os.walk(root):
            for f in files:
                if f == ".gitkeep":
                    continue
                z.write(os.path.join(dp, f))
print(f"bundled all results -> {ZIP} ({os.path.getsize(ZIP) // 1024} KB)")

# On Colab this triggers a browser download of the full bundle.
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    print("(not on Colab — grab the zip from the file browser on the left)")
    #trying

# Optional: copy everything to Google Drive instead of downloading.
# from google.colab import drive; drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/WSA_FerrariLuce" && cp -r data figures "/content/drive/MyDrive/WSA_FerrariLuce/"

bundled all results -> WSA_FerrariLuce_results.zip (8985 KB)
(not on Colab — grab the zip from the file browser on the left)
